# Load libraries

In [3]:
from pathlib import Path
import numpy as np
import trimesh
import pymeshlab
from pymeshlab import PercentageValue  # Older versions use 'from pymeshlab import Percentage'
import os

# Papermill parameters to be changed

In [4]:
# pore_directory
filepath = 'primitive_scaffold/left_L'

In [5]:
# Convert the string path to a Path object
pore_directory = Path(filepath)

# Use the / operator to concatenate the file path
pores_filepath = pore_directory / '1_pores_fill-holes.stl'

In [6]:
Specifications_file_path = pore_directory / "User_Specifications.txt"

# <span style="color:green"> Cleaning and repairing

#### Transform (rotate + translate) the pores 3D model

In [7]:
pores_mesh = trimesh.exchange.load.load(pores_filepath)

<span style="color:blue">Get original direction of the cylinder and set a target direction after transformation

In [8]:
initial_direction = pores_mesh.bounding_cylinder.direction
target_direction = [0,0,1] # target is the z axis

<span style="color:blue">Compute:

1. the angle between initial and target vectors
2. the dot product between initial and target vectors

<span style="color:blue">To build the rotation matrix

In [9]:
angle_vectors = trimesh.transformations.angle_between_vectors(initial_direction,target_direction)
product_vectors = trimesh.transformations.vector_product(initial_direction,target_direction)

In [10]:
rot_matrix = trimesh.transformations.rotation_matrix(angle_vectors, product_vectors)

Complete the rotation following the [trimesh example](https://trimsh.org/trimesh.transformations.html)

In [11]:
v2 = np.dot(initial_direction, rot_matrix[:3,:3].T)

In [12]:
np.allclose(trimesh.transformations.unit_vector(target_direction), trimesh.transformations.unit_vector(v2))

True

In [13]:
pores_mesh.apply_transform(rot_matrix)

<trimesh.Trimesh(vertices.shape=(2913381, 3), faces.shape=(5826128, 3), name=`1_pores_fill-holes.stl`)>

In [14]:
# move the centroid of the model to the center 
T = trimesh.transformations.translation_matrix([-pores_mesh.centroid[0], -pores_mesh.centroid[1], -pores_mesh.centroid[2]])
T

array([[ 1.        ,  0.        ,  0.        ,  3.40218251],
       [ 0.        ,  1.        ,  0.        ,  3.44963626],
       [ 0.        ,  0.        ,  1.        , -4.41372423],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [15]:
pores_mesh.apply_transform(T)

<trimesh.Trimesh(vertices.shape=(2913381, 3), faces.shape=(5826128, 3), name=`1_pores_fill-holes.stl`)>

In [16]:
pores_mesh.centroid

array([-8.88426445e-14, -2.22425898e-13, -4.09621527e-13])

#### Export stl file with new direction

In [17]:
trimesh.exchange.export.export_mesh(pores_mesh, filepath + '/1_pores_automatic_rotation.stl');

#### Create folder for Clean and repair the pores 3D model

In [18]:
import_model_filepath = filepath + '/1_pores_automatic_rotation.stl'

path = Path(filepath + '/cleaned')
path.mkdir(parents=True, exist_ok=True)
save_model_filepath = filepath + '/cleaned/pores_automatic_rotation.stl'

#### Create a new MeshSet

In [19]:
ms = pymeshlab.MeshSet()

#### load a new mesh in the MeshSet, and sets it as current mesh

In [20]:
ms.load_new_mesh(import_model_filepath)

#### Check if number of meshes in ms (MeshSet) is 1, if not, print it [it should be 1]

In [21]:
try:
    assert ms.mesh_number() == 1
except AssertionError as error:
  print("number of meshes is ",ms.mesh_number())

### Perform cleaning and repairing using the following filters and save after each filter

#### Remove disconnected objects like speckels (increase percentage to remove bigger speckels)

In [22]:
ms.apply_filter('meshing_remove_connected_component_by_diameter',mincomponentdiag = PercentageValue(20), removeunref = True)

{}

In [23]:
ms.save_current_mesh(filepath + '/cleaned/2_remove_speckels.stl')

#### Remove thin (needle-like) edges

In [24]:
# depth 10 is optimized (more doesnt do much but takes longer)
ms.apply_filter('generate_surface_reconstruction_screened_poisson', preclean = True, confidence = False, depth = 8 ,scale = 1.1)

{}

In [25]:
ms.save_current_mesh(filepath + '/cleaned/3_screened_poisson.stl')

#### Triangle remesh (reduce triangles due to memory restrictions in OF CFD)

In [26]:
# this does not take into account the topology of the sample so i changed it and i can specify number of triangles
# ms.apply_filter('meshing_isotropic_explicit_remeshing', targetlen = Percentage(0.5))

In [27]:
#ms.meshing_decimation_quadric_edge_collapse(targetfacenum=1000000, preserveboundary=True, preservenormal=True, preservetopology=True, autoclean=True, qualitythr=0.7, planarweight=0.0001 )

In [28]:
#ms.save_current_mesh(filepath + '/cleaned/4_remesh.stl')

#### Smoothening (sometimes makes more needles so check after because 3 might be too much)

In [29]:
ms.apply_filter('apply_coord_laplacian_smoothing', stepsmoothnum = 0,cotangentweight = False)

{}

In [30]:
ms.save_current_mesh(filepath + '/cleaned/5_smooth.stl')

#### Other filters

In [31]:
ms.apply_filter('meshing_merge_close_vertices')
ms.apply_filter('meshing_remove_duplicate_faces')
ms.apply_filter('meshing_remove_duplicate_vertices')
ms.apply_filter('meshing_remove_folded_faces')
ms.apply_filter('meshing_remove_null_faces')
ms.apply_filter('meshing_remove_t_vertices', method = 'Edge Collapse', threshold = 50)
ms.apply_filter('meshing_remove_unreferenced_vertices')
ms.apply_filter('meshing_remove_vertices_by_scalar')
ms.apply_filter('meshing_remove_null_faces')
ms.apply_filter('meshing_repair_non_manifold_edges', method = 'Remove Faces')

{}

#### Select then delete self-intersecting faces and print the number of faces that were intersecting then close the holes

In [32]:
ms.apply_filter('compute_selection_by_self_intersections_per_face')
old_int_face = "self intersecting faces were " , ms.current_mesh().selected_face_number()
print(old_int_face)

ms.apply_filter('meshing_remove_selected_faces')
new_int_face = " now they are " , ms.current_mesh().selected_face_number()
print( new_int_face)

holes = ms.apply_filter('meshing_close_holes', newfaceselected = True)
print (holes)

('self intersecting faces were ', 0)
(' now they are ', 0)
{'closed_holes': 2, 'new_faces': 5}


In [33]:
ms.save_current_mesh(filepath + '/cleaned/6_remove_self-intersecting_faces.stl')

#### AGAIN Remove disconnected objects like speckels (increase percentage to remove bigger speckels)

In [34]:
ms.apply_filter('meshing_remove_connected_component_by_diameter',mincomponentdiag = PercentageValue(20), removeunref = True)

{}

In [35]:
ms.save_current_mesh(filepath + '/cleaned/7_remove_disconneceted.stl')

### Save txt file of all filters that were used here and there values

In [36]:
ms.save_filter_script(filepath + '/used_MashLab_filters.txt')

In [37]:
# Add the self intersecting faces and holes that were removed to the txt file
with open(filepath + '/used_MashLab_filters.txt', 'a') as file:
    content = f'\n\n\n\n :{old_int_face} \n :{new_int_face} \n :{holes}' # Replace with the content you want to append
    file.write(content)

### Compute the porosity of the segment (volume of segment / volume of sylinder ROI)

In [38]:
pores_mesh = trimesh.load(filepath + '/cleaned/6_remove_self-intersecting_faces.stl')
pores_volume = pores_mesh.volume

cylinder_mesh = trimesh.load(filepath + '/cylinder_ROI.stl')
cylinder_volume = cylinder_mesh.volume

porosity = pores_volume/cylinder_volume

In [39]:
# Combine variable names and values into a dictionary
specifications = {
    "\n\nporosity of the entire section (CEP + VB + outside if taken)= volume_pores/volume_ROI": porosity,
    "%porosity": porosity*100,
}

# Open the file in write mode
with open(Specifications_file_path, "a") as file:
    # Write each variable as "variable_name: variable"
    for name, value in specifications.items():
        file.write(f"{name}: {value}\n")

### Move model to position 0,0,0

In [40]:
# Get the current center coordinates of the mesh
current_center = pores_mesh.centroid

# Calculate the translation vector to move the mesh to the origin
translation_vector = -current_center

# Perform the translation operation to move the mesh to (0, 0, 0)
pores_mesh.apply_translation(translation_vector)

pores_mesh.export(filepath + '/cleaned/8_centered.stl');

center_x, center_y, center_z = pores_mesh.centroid